In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix, f1_score
)

RANDOM_STATE = 2137

In [9]:
df = pd.read_csv("data_for_classifier.csv")
df = df.drop(columns = ["datetime", "ip_subnet", "postcode"])
df = df.loc[:, ~df.columns.str.startswith("merchant_info.")]
categorical_cols = ["type", "category", "payment_method", "city_state", "device"]
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
temporal_cols = ["orig_time_since_last_tx_as_orig", "orig_time_since_last_tx_as_dest", "dest_time_since_last_tx_as_orig", "dest_time_since_last_tx_as_dest"]
df[temporal_cols] = df[temporal_cols].fillna(99999999)
df = df.fillna(0)

In [10]:
features = [col for col in df.columns if col not in ["is_fraud", "orig", "dest"]]
binary_features = [col for col in features if df[col].isin([0, 1]).all()]
continuous_features = [col for col in features if col not in binary_features]
preprocessor = ColumnTransformer(
    transformers=[("num", StandardScaler(), continuous_features)], 
    remainder="passthrough" 
)

In [11]:
split_idx = int(len(df) * 0.9)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:]

X_train = train.drop(columns = ["is_fraud", "orig", "dest"])
y_train = train["is_fraud"]

X_test = test.drop(columns = ["is_fraud", "orig", "dest"])
y_test = test["is_fraud"]

In [12]:
val_split_idx = int(len(train) * (8/9))

X_train_sub = X_train[:val_split_idx]
y_train_sub = y_train[:val_split_idx]

X_val = X_train[val_split_idx:]
y_val = y_train[val_split_idx:]

In [13]:
# Scale features and weight the rare positive class during fitting.
model = Pipeline([
    ("preprocessor", preprocessor),
    ("logistic_regression", LogisticRegression(
        class_weight = "balanced",
        max_iter = 1000,
        random_state = RANDOM_STATE,
    )),
])

# Choose the alert threshold on the validation slice only.
model.fit(X_train_sub, y_train_sub)
val_scores = model.predict_proba(X_val)[:, 1]
thresholds = np.unique(np.quantile(val_scores, np.linspace(0, 1, 501)))
val_f1 = [f1_score(y_val, val_scores >= threshold) for threshold in thresholds]
best_threshold = float(thresholds[int(np.argmax(val_f1))])
print(f"Validation-selected threshold: {best_threshold}")
print(f"Validation F1: {max(val_f1)}")

# Refit on all pre-test rows and evaluate the held-out latest 10% once.
model.fit(X_train, y_train)
# Export here: weights and best_threshold
test_scores = model.predict_proba(X_test)[:, 1]
y_pred = (test_scores >= best_threshold).astype(int)
print("Test classification report:")
print(classification_report(y_test, y_pred))

Validation-selected threshold: 0.9943318711187326
Validation F1: 0.6785714285714286
Test classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    108833
           1       0.73      0.91      0.81       185

    accuracy                           1.00    109018
   macro avg       0.87      0.96      0.91    109018
weighted avg       1.00      1.00      1.00    109018



In [14]:
from pathlib import Path
import joblib

export_dir = Path("model_exports")
export_dir.mkdir(exist_ok=True)
test_keys = pd.read_csv(
    "data_for_classifier.csv",
    usecols=["datetime", "orig", "dest", "is_fraud"],
    parse_dates=["datetime"],
).iloc[-len(y_test):].reset_index(drop=True)
joblib.dump({
    "model": model,
    "X_test": X_test,
    "test_keys": test_keys,
    "threshold": best_threshold,
    "validation_f1": max(val_f1),
    "importance": pd.DataFrame({
        "feature": model.named_steps["preprocessor"].get_feature_names_out(),
        "coefficient": model.named_steps["logistic_regression"].coef_[0],
    }),
}, export_dir / "logistic_regression.joblib")

['model_exports/logistic_regression.joblib']